# MiniGPT: Attention Inspection and Generation

Load a trained MiniGPT checkpoint and inspect attention patterns, generate text.

In [ ]:
import sys

sys.path.insert(0, "src")

import torch
import matplotlib.pyplot as plt
import seaborn as sns

from from_zero.gpt import CharTokenizer, MiniGPT, GPTModelConfig, load_gpt_checkpoint, generate_text

## 1. Load Checkpoint and Model

In [ ]:
# Find latest checkpoint
import glob

checkpoints = sorted(glob.glob("artifacts/experiments/mini_gpt_shakespeare*/checkpoint/best.pt"))
if not checkpoints:
    checkpoints = sorted(glob.glob("artifacts/experiments/mini_gpt_*/checkpoint/best.pt"))

if checkpoints:
    latest = checkpoints[-1]
    print(f"Loading: {latest}")
    model, tokenizer, payload = load_gpt_checkpoint(latest)
    print(f"Step: {payload['step']}")
    print(f"Val loss: {payload.get('best_val_loss', 'N/A')}")
    print(f"Vocab size: {tokenizer.vocab_size}")
    print(f"Parameters: {model.num_parameters():,}")
else:
    print("No checkpoint found. Train first with: python scripts/train_gpt.py")

## 2. Attention Heatmap

In [ ]:
if checkpoints:
    # Get attention weights for a prompt
    prompt = "FIRST CITIZEN:\n"
    input_ids = tokenizer.encode(prompt)
    input_tensor = torch.tensor([input_ids])

    model.eval()
    with torch.no_grad():
        logits, _, attentions = model(input_tensor, return_attn=True)

    print(f"Input tokens: {input_ids}")
    print(f"Decoded: {tokenizer.decode(input_ids)}")
    print(f"Number of layers: {len(attentions)}")
    print(f"Attention shape per layer: {attentions[0].shape}")  # (B, H, T, T)

    # Plot attention for each layer
    fig, axes = plt.subplots(1, len(attentions), figsize=(4 * len(attentions), 3))
    if len(attentions) == 1:
        axes = [axes]

    for layer_idx, (ax, attn) in enumerate(zip(axes, attentions)):
        # Average over heads
        attn_avg = attn[0].mean(0).cpu().numpy()  # (T, T)
        im = ax.imshow(attn_avg, cmap="Blues", vmin=0, vmax=1)
        ax.set_title(f"Layer {layer_idx} (avg over heads)")
        ax.set_xlabel("Key position")
        ax.set_ylabel("Query position")

        # Add token labels
        tokens = [tokenizer.decode([tid]) for tid in input_ids]
        ax.set_xticks(range(len(tokens)))
        ax.set_yticks(range(len(tokens)))
        ax.set_xticklabels(tokens, rotation=90, fontsize=8)
        ax.set_yticklabels(tokens, fontsize=8)

    plt.tight_layout()
    plt.show()

## 3. Causal Mask Verification

In [ ]:
if checkpoints:
    # Verify upper triangle is zero (causal)
    for layer_idx, attn in enumerate(attentions):
        attn_avg = attn[0].mean(0)
        upper = torch.triu(attn_avg, diagonal=1)
        max_upper = upper.max().item()
        print(f"Layer {layer_idx}: max upper-triangle attention = {max_upper:.2e}")
        assert max_upper < 1e-6, f"Causal mask violated in layer {layer_idx}!"

## 4. Text Generation

In [ ]:
if checkpoints:
    prompts = ["FIRST CITIZEN:\n", "ROMEO:\n", "To be, or not to be", "HAMLET:"]

    for prompt in prompts:
        print(f"\n{'=' * 50}")
        print(f"Prompt: {repr(prompt)}")
        generated = generate_text(
            model, tokenizer, prompt, max_new_tokens=200, temperature=0.8, top_k=40, seed=42
        )
        print(generated)

## 5. Temperature and Top-k Effects

In [ ]:
if checkpoints:
    prompt = "ROMEO:\n"

    print("Temperature comparison:")
    for temp in [0.2, 0.5, 0.8, 1.0, 1.5]:
        gen = generate_text(
            model, tokenizer, prompt, max_new_tokens=80, temperature=temp, top_k=40, seed=42
        )
        print(f"  T={temp}: {gen[:100]}...")

    print("\nTop-k comparison (T=0.8):")
    for k in [1, 5, 20, 40, 0]:  # 0 = no top-k
        gen = generate_text(
            model,
            tokenizer,
            prompt,
            max_new_tokens=80,
            temperature=0.8,
            top_k=k if k > 0 else None,
            seed=42,
        )
        label = f"k={k}" if k > 0 else "no top-k"
        print(f"  {label}: {gen[:100]}...")

## 6. Token Embeddings Visualization

In [ ]:
if checkpoints:
    # Get token embeddings
    emb = model.token_embedding.weight.data.cpu().numpy()  # (V, d)

    # PCA for 2D visualization
    from sklearn.decomposition import PCA

    pca = PCA(n_components=2)
    emb_2d = pca.fit_transform(emb)

    plt.figure(figsize=(10, 8))
    chars = tokenizer.chars
    for i, ch in enumerate(chars):
        plt.scatter(emb_2d[i, 0], emb_2d[i, 1], alpha=0.7)
        plt.annotate(repr(ch), (emb_2d[i, 0], emb_2d[i, 1]), fontsize=8)

    plt.title("Token Embeddings (PCA to 2D)")
    plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(alpha=0.3)
plt.show()